In [1]:
!pip -q install catboost pyarrow
import catboost, torch
print("catboost:", catboost.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.2 MB/s eta 0:00:00
catboost: 1.2.10
CUDA: True NVIDIA A100-SXM4-40GB
Mounted at /content/drive


In [2]:
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/ct_agent/staging")
man = json.loads((BASE / "staging_manifest.json").read_text())

def sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

for name, expect in man["files_sha256"].items():
    got = sha256(BASE / name)
    assert got == expect, f"{name}: sha mismatch {got[:16]} != {expect[:16]}"
print("sha256 ok:", list(man["files_sha256"]))

tr = pd.read_parquet(BASE / "tune_train.parquet")
va = pd.read_parquet(BASE / "tune_valid.parquet")
scales = json.loads((BASE / "scales_tune.json").read_text())
feat_cols = man["feature_columns"]
cat_cols = man["categorical"]
y_cols = [f"y_raw_h{j}" for j in range(1, 29)]
assert len(tr) == man["rows"]["tune_train"] and len(va) == man["rows"]["tune_valid"]

def split(df):
    X = df[feat_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype(int)
    s = df["series_id"].map(scales).to_numpy(np.float64)
    assert np.isfinite(s).all() and (s > 0).all()
    y = df[y_cols].to_numpy(np.float64) / s[:, None]
    return X, y

Xtr, ytr = split(tr)
Xva, yva = split(va)
print(f"train={len(Xtr)} valid={len(Xva)} features={len(feat_cols)} cats={cat_cols}")

sha256 ok: ['tune_train.parquet', 'tune_valid.parquet', 'scales_tune.json']
train=170096 valid=21714 features=23 cats=['store_id', 'dept_id', 'cat_id', 'state_id']


In [3]:
import time
from catboost import CatBoostRegressor, Pool

order = man["grid_order_full"]
p0 = order[0]
print("permutation first combo:", p0)

tr_pool = Pool(Xtr, ytr, cat_features=cat_cols)
va_pool = Pool(Xva, yva, cat_features=cat_cols)

probe = CatBoostRegressor(
    loss_function="MultiRMSE", eval_metric="MultiRMSE",
    task_type="GPU", devices="0",
    iterations=50, depth=p0["depth"],
    learning_rate=p0["learning_rate"], l2_leaf_reg=p0["l2_leaf_reg"],
    random_seed=man["model_seed"], verbose=25, allow_writing_files=False,
)
t0 = time.perf_counter()
probe.fit(tr_pool, eval_set=va_pool)
t50 = time.perf_counter() - t0
print(f"50 iters: {t50:.1f}s -> ~{t50/50*1000:.0f}s per 1000 iters (upper bound)")

permutation first combo: {'trial_id': 21, 'depth': 10, 'learning_rate': 0.06, 'l2_leaf_reg': 1.0}
0:	learn: 5.5417760	test: 5.2810476	best: 5.2810476 (0)	total: 172ms	remaining: 8.43s
25:	learn: 5.3417824	test: 5.1478595	best: 5.1478595 (25)	total: 3.44s	remaining: 3.18s
49:	learn: 5.2786103	test: 5.1074973	best: 5.1074973 (49)	total: 6.61s	remaining: 0us
bestTest = 5.107497262
bestIteration = 49
50 iters: 7.5s -> ~150s per 1000 iters (upper bound)


In [4]:
T_BUDGET = man["T_budget_s"]
MAX_ITERS, ES = 1000, 50

def run_trial(p):
    m = CatBoostRegressor(
        loss_function="MultiRMSE", eval_metric="MultiRMSE",
        task_type="GPU", devices="0",
        iterations=MAX_ITERS, depth=p["depth"],
        learning_rate=p["learning_rate"], l2_leaf_reg=p["l2_leaf_reg"],
        random_seed=man["model_seed"], early_stopping_rounds=ES,
        use_best_model=True, verbose=0, allow_writing_files=False,
    )
    t0 = time.perf_counter()
    m.fit(tr_pool, eval_set=va_pool)
    sec = time.perf_counter() - t0
    score = float(m.get_best_score()["validation"]["MultiRMSE"])
    return m, score, int(m.get_best_iteration()), sec

m1, score1, it1, t1 = run_trial(p0)
n_run = min(len(order), max(1, int(T_BUDGET // t1)))
print(f"trial 1: score={score1:.6f} best_iter={it1} t1={t1:.1f}s -> n_run={n_run}")
algo_params = {k: m1.get_all_params().get(k) for k in
               ("boosting_type", "bootstrap_type", "subsample",
                "random_strength", "border_count", "leaf_estimation_method")}
print("gpu algo params:", algo_params)

trial 1: score=4.972009 best_iter=947 t1=137.4s -> n_run=13
gpu algo params: {'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'subsample': None, 'random_strength': 1, 'border_count': 128, 'leaf_estimation_method': 'Newton'}


In [5]:
results = [{"trial": p0["trial_id"], "order_pos": 0, **p0,
            "score": score1, "best_iteration": it1,
            "sec_total": round(t1, 1)}]
for pos in range(1, n_run):
    p = order[pos]
    _, sc, bi, sec = run_trial(p)
    results.append({"trial": p["trial_id"], "order_pos": pos, **p,
                    "score": sc, "best_iteration": bi,
                    "sec_total": round(sec, 1)})
    print(f"{pos+1}/{n_run} id={p['trial_id']} score={sc:.6f} "
          f"best_iter={bi} sec={sec:.1f}")

finite = [r for r in results if np.isfinite(r["score"])]
best = min(finite, key=lambda r: (r["score"], r["order_pos"]))
print("selected:", best)

2/13 id=14 score=4.989088 best_iter=997 sec=41.6
3/13 id=0 score=5.036504 best_iter=993 sec=20.8
4/13 id=5 score=5.020673 best_iter=998 sec=20.4
5/13 id=25 score=4.978231 best_iter=706 sec=104.7
6/13 id=12 score=4.968522 best_iter=996 sec=44.0
7/13 id=6 score=4.979622 best_iter=994 sec=21.1
8/13 id=19 score=4.985887 best_iter=997 sec=134.5
9/13 id=15 score=4.971513 best_iter=868 sec=40.8
10/13 id=26 score=4.979698 best_iter=853 sec=122.0
11/13 id=8 score=4.999313 best_iter=998 sec=20.3
12/13 id=24 score=5.000009 best_iter=287 sec=47.1
13/13 id=2 score=5.058410 best_iter=999 sec=20.3
selected: {'trial': 12, 'order_pos': 5, 'trial_id': 12, 'depth': 8, 'learning_rate': 0.06, 'l2_leaf_reg': 1.0, 'score': 4.96852167460518, 'best_iteration': 996, 'sec_total': 44.0}


In [6]:
out = {
    "protocol": "ST6-12",
    "model": "catboost_multirmse_h28",
    "compute": {"platform": "colab", "task_type": "GPU",
                "gpu": torch.cuda.get_device_name(0),
                "gpu_probe_50iter_s": round(t50, 1),
                "gpu_algo_params": algo_params,
                "note": ("GPU training is not bit-reproducible across "
                         "runs/devices; artifacts are frozen by sha256 "
                         "instead (TSMixer checkpoint convention)")},
    "grid": man["grid"], "grid_seed": man["grid_seed"],
    "model_seed": man["model_seed"],
    "execution_order": man["execution_order"],
    "budget_rule": {"T_budget_s": T_BUDGET,
                    "budget_source": man["budget_source"],
                    "t_trial1_s": round(t1, 1), "n_run": n_run},
    "max_iters": MAX_ITERS, "early_stopping": ES,
    "staging_manifest_sha": {k: v[:16] for k, v in man["files_sha256"].items()},
    "spec_hash": man["spec_hash"], "encoder_hash": man["encoder_hash"],
    "cpu_measurement_record": man["cpu_measurement_record"],
    "trials": results,
    "selected": {k: best[k] for k in
                 ("trial_id", "depth", "learning_rate", "l2_leaf_reg",
                  "score", "best_iteration")},
}
apto_dir = BASE.parent / "apto"
apto_dir.mkdir(exist_ok=True)
(apto_dir / "stage6_apto_tuning.json").write_text(json.dumps(out, indent=1))
print("written", apto_dir / "stage6_apto_tuning.json")

written /content/drive/MyDrive/ct_agent/apto/stage6_apto_tuning.json


In [7]:
sel = best  # cell 5
m_cal = CatBoostRegressor(
    loss_function="MultiRMSE", eval_metric="MultiRMSE",
    task_type="GPU", devices="0",
    iterations=MAX_ITERS, depth=sel["depth"],
    learning_rate=sel["learning_rate"], l2_leaf_reg=sel["l2_leaf_reg"],
    random_seed=man["model_seed"], early_stopping_rounds=ES,
    use_best_model=True, verbose=0, allow_writing_files=False,
)
m_cal.fit(tr_pool, eval_set=va_pool)
cal_best = int(m_cal.get_best_iteration())
cal_score = float(m_cal.get_best_score()["validation"]["MultiRMSE"])
print(f"calibration refit: best_iter={cal_best} score={cal_score:.6f} "
      f"(trial run was {sel['best_iteration']}/{sel['score']:.6f}; GPU "
      f"nondeterminism makes small drift normal)")

yhat_scaled = m_cal.predict(va_pool)          # (n, 28), scaled space
s_va = va["series_id"].map(scales).to_numpy(np.float64)
yhat_raw = yhat_scaled * s_va[:, None]

pred = pd.DataFrame({"series_id": va["series_id"].to_numpy(),
                     "origin_idx": va["origin_idx"].to_numpy()})
for j in range(28):
    pred[f"yhat_raw_h{j+1}"] = yhat_raw[:, j]
    pred[f"y_raw_h{j+1}"] = va[f"y_raw_h{j+1}"].to_numpy(np.float64)
pred_path = apto_dir / "calibration_valid_predictions.parquet"
pred.to_parquet(pred_path, index=False)

model_path = apto_dir / "apto_calibration.cbm"
m_cal.save_model(str(model_path))

import hashlib
def fsha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

side = {"calibration_refit": {"best_iteration": cal_best,
                              "score": cal_score,
                              "trial_reference": sel},
        "files_sha256": {pred_path.name: fsha(pred_path),
                         model_path.name: fsha(model_path)}}
(apto_dir / "calibration_manifest.json").write_text(json.dumps(side, indent=1))
print("written:", pred_path.name, model_path.name, "sha:",
      {k: v[:16] for k, v in side["files_sha256"].items()})

calibration refit: best_iter=999 score=4.966726 (trial run was 996/4.968522; GPU nondeterminism makes small drift normal)
written: calibration_valid_predictions.parquet apto_calibration.cbm sha: {'calibration_valid_predictions.parquet': '40f9a17763f74f0d', 'apto_calibration.cbm': 'f8fb7f247e956c6f'}


In [9]:
from catboost import CatBoostRegressor, Pool
import hashlib, json, time
import numpy as np
import pandas as pd

fman = json.loads((BASE / "fold_staging_manifest.json").read_text())

def fsha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

for name, expect in fman["files_sha256"].items():
    got = fsha(BASE / name)
    assert got == expect, f"{name}: {got[:16]} != {expect[:16]}"
print("fold staging sha256 ok:", len(fman["files_sha256"]), "files")

tun = json.loads((apto_dir / "stage6_apto_tuning.json").read_text())
sel = tun["selected"]
FIXED_ITERS = int(sel["best_iteration"])
print("HP:", sel, "-> fixed iterations", FIXED_ITERS)

fold_out = {}
for k in (1, 4, 7, 10):
    df = pd.read_parquet(BASE / f"fold_train_f{k}.parquet")
    sc = json.loads((BASE / f"scales_f{k}.json").read_text())
    X = df[feat_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype(int)
    s = df["series_id"].map(sc).to_numpy(np.float64)
    assert np.isfinite(s).all() and (s > 0).all()
    y = df[[f"y_raw_h{j}" for j in range(1, 29)]].to_numpy(np.float64) / s[:, None]
    assert len(df) == fman["folds"][str(k)]["rows"]

    m = CatBoostRegressor(
        loss_function="MultiRMSE", task_type="GPU", devices="0",
        iterations=FIXED_ITERS, depth=sel["depth"],
        learning_rate=sel["learning_rate"], l2_leaf_reg=sel["l2_leaf_reg"],
        random_seed=tun["model_seed"], verbose=0, allow_writing_files=False,
    )
    t0 = time.perf_counter()
    m.fit(Pool(X, y, cat_features=cat_cols))
    sec = time.perf_counter() - t0
    mp = apto_dir / f"apto_f{k}.cbm"
    m.save_model(str(mp))
    fold_out[str(k)] = {"rows": len(df), "sec_total": round(sec, 1),
                        "tree_count": int(m.tree_count_),
                        "model_sha256": fsha(mp)}
    print(f"fold {k}: rows={len(df)} sec={sec:.1f} trees={m.tree_count_} "
          f"sha={fold_out[str(k)]['model_sha256'][:16]}")

fold_manifest = {
    "protocol": "ST6-12 (6)",
    "hp": sel, "fixed_iterations": FIXED_ITERS,
    "model_seed": tun["model_seed"],
    "gpu": tun["compute"]["gpu"],
    "no_eval_set": "fold retraining uses fixed iterations, no validation (ST3-03 (d))",
    "staging_sha": {k: v[:16] for k, v in fman["files_sha256"].items()},
    "folds": fold_out,
}
(apto_dir / "fold_models_manifest.json").write_text(
    json.dumps(fold_manifest, indent=1))
print("written fold_models_manifest.json")

fold staging sha256 ok: 8 files
HP: {'trial_id': 12, 'depth': 8, 'learning_rate': 0.06, 'l2_leaf_reg': 1.0, 'score': 4.96852167460518, 'best_iteration': 996} -> fixed iterations 996
fold 1: rows=195968 sec=43.5 trees=996 sha=0aa343d47e3e0723
fold 4: rows=208904 sec=44.1 trees=996 sha=451f34586a471ccc
fold 7: rows=221840 sec=44.6 trees=996 sha=5170ec1c9bdc5155
fold 10: rows=234776 sec=45.2 trees=996 sha=c75b7dcde50c82ba
written fold_models_manifest.json


In [23]:
mutd = 1